In [ ]:
#here are the packages I used to do this. Feel free to add to this as needed
import astropy.units as u
from astropy.io import (fits, ascii)
from scipy.optimize import curve_fit
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import trapezoid
import pandas as pd
import os, glob

In [ ]:
#set paths
photometry_path = r'../files/'
temperature_path = r'../files/' #model grid
filter_path = r'../files/filter_response/' # filter response function path
model_path = r'../SED_Models/' #sed models
s
#READ CSV
phot_data = pd.read_csv(photometry_path + r'photometry.csv')
temp_grid = pd.read_csv(temperature_path + r'Temperature_Grid.csv')
print('--------------------------------------------------------------')
print(phot_data.columns.to_list())
print('-------------------------------------------------------------')
print(temp_grid.columns.to_list())
print('----------------------------filter------------------------------')
print(phot_data['Filter'])

filter_list = list(np.array(phot_data['Filter'], dtype = str))

In [ ]:
phot_data

In [ ]:
modeling_data = phot_data[0:-1][['Filter', 'wav_eff']]

filter_files = np.array([r'2MASS\2MASS_2MASS.J.dat', r'2MASS\2MASS_2MASS.H.dat', r'2MASS\2MASS_2MASS.Ks.dat',
               r'WISE\WISE_WISE.W1.dat', r'WISE\WISE_WISE.W2.dat',r'WISE\WISE_WISE.W3.dat',r'WISE\WISE_WISE.W4.dat',
               r'Gaia\GAIA_GAIA3.Gbp.dat', r'Gaia\GAIA_GAIA3.G.dat',r'Gaia\GAIA_GAIA3.Grp.dat',
               r'GALEX\GALEX_GALEX.FUV.dat', r'GALEX\GALEX_GALEX.NUV.dat'])
filter_names = np.array(modeling_data['Filter'])

In [ ]:
def star_flux(mag, zero_point, m0=0):
    flux = 10**(-(mag - m0)/2.5) * zero_point
    return flux

def filter_magnitude(flux, zero_point, m0 = 0):
    m = -2.5*np.log10(flux/zero_point) + m0
    return m

def model_flux_loader(model_file, logg):
    '''
    filter_file (str) - path of the filter transmission file
    model_file (str) - path of ATLAS9 model file
    logg (str) - surface gravity
    '''
    model_path =  model_file
    model_data = fits.open(model_path)[1].data
    wav = model_data['WAVELENGTH']
    model_flux = model_data[logg]
    return wav, model_flux

def filter_transmission(filter_file, wav, model_flux, return_original = False):
    filter_file = filter_file
    filt_data = np.genfromtxt(filter_file, dtype= float)
    filt_wav = np.array(filt_data[:,0], dtype = float)
    filt_flux = np.array(filt_data[:,1], dtype = float)
    
    print(len(filt_flux) == len(filt_wav))
    
    if return_original == True:
        return filt_wav, filt_flux
    else:
        filt_interflux = np.interp(wav, filt_wav, filt_flux) 
        return filt_interflux
    
power_law = lambda x, a, k: a*(x**(-k))

def model_filt_flux(filt_name, filter_file, wav, flux, 
                    inter_wav, inter_flux, rad = 1):
    #Get filter data
    filt_index = filter_list.index(filt_name)
    
    cent_wav = float(phot_data['wav_eff'][filt_index])
    W_eff = float(phot_data['Weff'][filt_index])
    
    
    #Open 
    filter_file = filter_file
    filt_data = np.genfromtxt(filter_file, dtype= float)
    filt_wav = np.array(filt_data[:,0], dtype = float)
    filt_flux = np.array(filt_data[:,1], dtype = float)
    
    #Normalization
    div = 1
    if max(filt_flux) > 1:
        div = max(filt_flux)
        
    #force ends to equal 0    
    if min(filt_flux) > 0:
        filt_flux[0] = 0
        filt_flux[-1] = 0

    filt_interflux = np.interp(inter_wav, filt_wav, filt_flux)/div #interpoloate
    av_flux = trapezoid(filt_interflux, x = inter_wav)/W_eff #calculate model flux
    
    #Find flux at central wavelength
    filt = np.where((inter_wav >= cent_wav - rad) & (inter_wav <= cent_wav + rad))  
    return av_flux * np.median(inter_flux[filt])
    
    
def get_flux_models(wavelength, model_flux, lim = 10**4.8):
    '''
    get flux models for a given SED 
    
    inputs:
    - wavelength (float array) - wavelength of SED MODEL
    - model_flux (array) - array of desired SED MODELs
    
    outputs:
    flux_list (array) - array of model fluxes
    '''
    #interpolate
    int_wav = np.arange(min(wavelength), max(wavelength), 1)
    int_model_flux = np.interp(int_wav, wavelength, model_flux)
    #do a curve fit
    (a1, k1), pcov = curve_fit(power_law, wavelength[wavelength > lim], model_flux[wavelength > lim], p0=[10e12, 3.42])
    int_model_flux[int_wav > lim]  = power_law(int_wav[int_wav > lim], a1, k1)
    

    #Find Names
    temp_list = np.array(list(zip(filter_files, filter_names)))
    flux_list = np.array([model_filt_flux(item[1], filter_path + item[0], wav = wavelength, flux = model_flux,
                                              inter_wav = int_wav, inter_flux = int_model_flux) for item in temp_list])
    return flux_list

In [ ]:
R_sun = 6.95700E8 #m
parsec = 3.0857E+16 #m

In [ ]:
Mag = -1
sec_temperature = 14000
temperature = 5000
dist = 10

logg = 3.0
i_logg = 'g' + str(int(logg*10))

#secondary parameters
logg_sec = np.array(temp_grid['logg'])[list(temp_grid['temperature']).index(sec_temperature)]
i_logg_sec = 'g' + str(int(logg_sec*10))

R_sec = np.array(temp_grid['radius'])[list(temp_grid['temperature']).index(sec_temperature)]
ms_flux_scale = ((R_sec * R_sun)/ (dist * parsec))**2

#primary model fluxes
wavelength, sed_flux = model_flux_loader(model_path + f'ckp00\ckp00_{temperature}.fits', 'g40')
g_band_flux = star_flux(Mag, zero_point=2.500000e-09, m0=0) #gaia gband flux (m=M)
giant_model_fluxes = get_flux_models(wavelength, sed_flux)#giant model fluxes

#secondary model fluxes

if sec_temperature < 6250:
    ms_file = model_path + f'ckp00\ckp00_{sec_temperature}.fits'
else:
    ms_file = model_path + f'hot_ms\ckp00_{sec_temperature}.fits'

const, ms_sed_flux = model_flux_loader(ms_file, i_logg_sec)
ms_model_fluxes = get_flux_models(wavelength, ms_sed_flux)#giant model fluxes

#flux scales
flux_scale = g_band_flux/giant_model_fluxes[8]
ms_flux_scale = ((R_sec * R_sun)/ (dist * parsec))**2

#combined flux
combined_sed = ms_sed_flux * ms_flux_scale + flux_scale * sed_flux
com_mod_fluxes = get_flux_models(wavelength, combined_sed)#giant model fluxes

#wavelengths
λeffs = np.array(modeling_data['wav_eff'])

In [ ]:
print(ms_flux_scale, flux_scale)

In [ ]:
filter_label = ['$J$', '$H$', '$K_{s}$', '$W1$', '$W2$', '$W3$', '$W4$',
                '$G_{BP}$', '$G$', '$G_{RP}$', '$FUV$', '$NUV$']
final_yorient = ['bottom', 'bottom', 'bottom',
                 'bottom', 'bottom', 'bottom', 'bottom',
                 'top', 'top', 'top', 'bottom', 'bottom']
final_xorient = ['left', 'left', 'left',
                 'left', 'left', 'left', 'left',
                 'right', 'left', 'left', 'left', 'left']

#Models
fig1, ax = plt.subplots(1, figsize = (6, 5))
#SED MODELS
ax.set_title(f'Model SED, $M_G$ = {Mag}')

ax.plot(wavelength, sed_flux * flux_scale, color = 'black',
        label = f'Primary SED (T = {temperature} K, logg = {logg})')

ax.plot(wavelength, ms_sed_flux * ms_flux_scale, color = 'slategrey',
        label = f'MS SED (T = {sec_temperature} K, logg = {logg_sec})')

ax.plot(wavelength, combined_sed, color = 'grey',
        label = 'Combined SED')

#MODEL FLUXES
ax.scatter(λeffs,  ms_model_fluxes  * ms_flux_scale, marker = 'o', 
           color = 'green', label = f'MS Model Flux', edgecolor='aquamarine', zorder=10)

ax.scatter(λeffs, com_mod_fluxes, marker = 'o', 
           color = 'orange', label = f'Combined Model Flux',zorder=9)

ax.scatter(λeffs, giant_model_fluxes *flux_scale, marker = 'P', 
           color = 'red', label = f'Primary Model Flux',zorder=9)


ax.set_xlabel('Wavelength (A)')
ax.set_ylabel('Flux $(erg/cm^{2}/s/A)$')
ax.set_yscale('log')
ax.set_xscale('log')

ax.legend(fontsize=9, framealpha=1).set_zorder(10)

max_flux = com_mod_fluxes[8]

coeff_low = np.ceil(np.log10(max_flux))+0.5
coeff_high = np.floor(np.log10(giant_model_fluxes[6] * flux_scale))-0.5

low_ylim = 10**coeff_low
high_ylim = 10**coeff_high


#adjust ylims
ax.set_xlim(10**3, 10**5.5)
ax.set_ylim(high_ylim, low_ylim) #10e-20, 10e-11)


# Calculate Magnitudes

In [ ]:
nuv = filter_magnitude(com_mod_fluxes[-1], zero_point=2.06e-16, m0=20.08)
fuv = filter_magnitude(com_mod_fluxes[-2], zero_point=1.40e-15, m0=18.82)
fuv_nuv = fuv-nuv
print(nuv, fuv, fuv_nuv)

# Save Contents

In [ ]:
save_mag_file = r'files//combined_nuv_fuv_flux.txt' #change path if needed

if not os.path.exists(save_mag_file):
    f = open(save_mag_file, 'a')
    print('M_G', 'giant_temp', 'giant_logg', 'ms_temp', 'fuv','nuv','fuv-nuv', file=f)
    print(Mag, temperature, logg, sec_temperature, fuv, nuv, fuv_nuv, file=f)
else:
    f = open(save_mag_file, 'a')
    print(Mag, temperature, logg, sec_temperature, fuv, nuv, fuv_nuv, file=f)

f.close()